<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/11-catalyst-optimizer/02-pushdown-pruning-folding.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 11.2 — Pushdown, pruning, folding: read it off the scan node

Watch constant folding delete arithmetic, projection pruning narrow
`ReadSchema`, and predicate pushdown fill `PushedFilters` — then watch
a Python UDF sabotage all three. Writes a small partitioned parquet
dataset to a local `output/` dir and self-cleans.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("11.2-pushdown-pruning-folding")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Set up a columnar, partitioned dataset on disk

Parquet so pruning can skip columns and row groups; `partitionBy("dt")`
so a date filter can skip whole directories.

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)

(spark.range(2_000_000)
 .withColumn("dt",  F.date_add(F.lit("2026-01-01"), (col("id") % 5).cast("int")))
 .withColumn("k",   (col("id") % 100).cast("int"))
 .withColumn("v",   col("id") * 2)
 .withColumn("note", F.concat(F.lit("row-"), col("id").cast("string")))
 .write.mode("overwrite").partitionBy("dt").parquet("output/events"))

evt = spark.read.parquet("output/events")
print(evt.columns)

## Constant folding: arithmetic evaluated once, at planning time

You write `60 * 60 * 24`; the optimized plan shows `86400`. The
multiply never runs per row.

In [ ]:
(evt.filter(col("v") > 60 * 60 * 24)
     .select("k")
     .queryExecution.optimizedPlan)   # look for Filter (v > 86400), not the product

## Projection pruning: `ReadSchema` shows only what you read

Select two of five columns; the scan reads two. `formatted` mode puts
`ReadSchema` in the scan's detail block.

In [ ]:
evt.select("k", "v").explain(mode="formatted")
# find ReadSchema: struct<k,v> — 'note', 'id', 'dt' bytes never read

## Predicate pushdown: `PushedFilters` + `PartitionFilters`

A value filter pushes into the parquet reader (`PushedFilters`); a
partition-column filter skips directories (`PartitionFilters`) — the
strongest pushdown, I/O that never happens.

In [ ]:
(evt.filter((col("dt") == "2026-01-03") & (col("v") > 500))
     .select("k")
     .explain(mode="formatted"))
# PartitionFilters: [dt = 2026-01-03]  (1 of 5 dirs read)
# PushedFilters:    [IsNotNull(v), GreaterThan(v, 500)]

## A Python UDF blinds the optimizer — pushdown AND pruning lost

The predicate becomes an opaque box: it can't push, and the columns it
touches can't be pruned. Compare the two scan nodes.

In [ ]:
from pyspark.sql.types import BooleanType

@F.udf(BooleanType())
def big_v(x):
    return x > 500

print("NATIVE — filter pushes to the scan:")
evt.filter(col("v") > 500).select("k").explain(mode="formatted")

print("\nUDF — filter stranded ABOVE a full scan (no PushedFilters):")
evt.filter(big_v(col("v"))).select("k").explain(mode="formatted")

## Prove the difference is real, not cosmetic

Time both. The UDF version reads every row into Python; the native
version lets parquet skip row groups. Same answer, very different work.

In [ ]:
import time
def timed(df, label):
    t = time.time(); n = df.count(); print(f"{label}: {n:,} rows in {time.time()-t:.2f}s")

timed(evt.filter(col("v") > 3_900_000).select("k"), "native  ")
timed(evt.filter(big_v(col("v"))).select("k"),      "udf     ")

In [ ]:
spark.stop()
shutil.rmtree("output", ignore_errors=True)   # self-clean

## Exercises

1. `evt.filter(col("v") > lit(1000) * 60).explain(mode="cost")` — find
   the folded constant. Now nest it deeper: `((v + 1) - 1) > 100`. Does
   the optimizer simplify `+1-1`? What rule is that?
2. Compare `ReadSchema` for `evt.select("k")` vs `evt.select("*")` vs
   `evt.groupBy("k").sum("v")`. Which columns does each actually read?
3. Filter on `dt` with a cast: `filter(col("dt").cast("string") ==
   "2026-01-03")`. Check `PartitionFilters` — did partition pruning
   still fire? Explain what you see.
4. Put the UDF in a `withColumn` instead of a filter:
   `evt.withColumn("flag", big_v(col("v"))).select("k")`. Does
   projection pruning of `v` survive? Why or why not?
5. Read the same data as CSV (write an unpartitioned CSV copy first).
   Filter on `v`. Does `PushedFilters` populate? Why can't CSV skip
   row groups the way parquet can?

In [ ]:
# your exercise code here